# 3. Probability Distributions — Modeling How Each Input Behaves

**Building a Heart Disease Risk-Screening System — Notebook 3 of 12, Stage 1: Understanding the Raw Signals**

Notebook 2 gave every input a mean and a spread. This notebook asks the sharper
question: what *shape* does each input's variation follow? Assume the wrong shape
and every downstream tool built on it — confidence intervals in Notebook 4,
hypothesis tests in Notebooks 6-8 — inherits a wrong assumption baked in from the
start.

## The topic

A **probability distribution** is a complete description of how likely each
possible value (or range of values) of a variable is — not just its center and
spread, but its full shape: symmetric or lopsided, bounded or unbounded, one
hump or several.

## Why it matters for this system

Statistical tests and confidence intervals are built on assumptions about
shape (most commonly Normality). Assumptions that don't hold don't just produce
slightly-off numbers — they can flip a "significant" result to "not significant"
or vice versa. Checking the shape *before* trusting a downstream tool is the single
highest-leverage habit this notebook builds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
print(df.shape)

## The toolkit

| Distribution | Fits variables that... |
|---|---|
| **Normal** | Cluster symmetrically around a center, unbounded in both directions |
| **Log-normal** | Are right-skewed and strictly positive (a log-transform makes them Normal) |
| **Bernoulli** | Take exactly two values (yes/no, disease/no-disease) |
| **Binomial** | Count successes across several independent Bernoulli trials |
| **Poisson** | Count rare, independent events over a fixed window |

## How to choose

1. **Plot the histogram.** Symmetric bell shape, lopsided tail, or something else
   entirely?
2. **Check skewness** (Notebook 2). Near zero → candidate for Normal. Strongly
   positive → try a log-transform and recheck.
3. **Q-Q plot against the candidate.** Points hugging the diagonal confirm the fit;
   systematic curving in the tails rejects it.
4. **Match the data-generating story**, not just the shape: a single yes/no →
   Bernoulli; a count of successes out of n independent trials → Binomial; rare
   events over time → Poisson.

## Applied to the registry

### Testing `thalach` (max heart rate) against a Normal curve

In [ ]:
mu, sigma = df["thalach"].mean(), df["thalach"].std()
print(f"thalach: mean={mu:.1f}, std={sigma:.1f}, skewness={stats.skew(df['thalach']):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(df["thalach"], bins=25, density=True, alpha=0.6, color="steelblue")
xs = np.linspace(df["thalach"].min(), df["thalach"].max(), 200)
axes[0].plot(xs, stats.norm.pdf(xs, mu, sigma), "r-", lw=2, label=f"Normal({mu:.0f}, {sigma:.0f})")
axes[0].set_title("thalach vs. a fitted Normal curve"); axes[0].legend()

stats.probplot(df["thalach"], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot -- points close to the line support the Normal fit")
plt.tight_layout(); plt.show()

### The empirical rule as a numeric check

For a genuinely Normal variable, ~68% of values fall within 1 standard deviation of
the mean, ~95% within 2. Verify this directly instead of assuming it.

In [ ]:
for k in [1, 2, 3]:
    within_k = ((df["thalach"] >= mu - k*sigma) & (df["thalach"] <= mu + k*sigma)).mean()
    theoretical = stats.norm.cdf(k) - stats.norm.cdf(-k)
    print(f"within {k} std dev: actual={within_k:.1%}   Normal theory={theoretical:.1%}")

### When Normal is the wrong fit: checking `chol`

Cholesterol has a hard floor near zero and no real ceiling — right-skew is a real
possibility worth checking before assuming Normality the way `thalach` seemed to
support.

In [ ]:
print(f"chol skewness: {stats.skew(df['chol']):.2f}  (compare to thalach's {stats.skew(df['thalach']):.2f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(df["chol"], bins=25, color="salmon", alpha=0.7)
axes[0].set_title("chol -- check for a right tail")
stats.probplot(df["chol"], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot for chol vs. Normal")
plt.tight_layout(); plt.show()

If skewness is noticeably positive and the Q-Q plot curves away from the diagonal
in the tails, a log-transform is the standard fix — re-check both diagnostics on
`np.log(df["chol"])` before deciding whether the transform is actually needed here,
or whether the raw values are close enough to proceed without one.

In [ ]:
log_chol = np.log(df["chol"])
print(f"log(chol) skewness: {stats.skew(log_chol):.2f}  (vs. {stats.skew(df['chol']):.2f} untransformed)")

### Bernoulli and Binomial: the disease outcome itself

`target` is a Bernoulli variable at the individual-patient level. Zoom out to "how
many of the next 20 screened patients will have disease" and you're asking a
Binomial question — exact probabilities, no simulation needed.

In [ ]:
p_disease = df["target"].mean()
n_next_patients = 20

outcomes = np.arange(0, n_next_patients + 1)
probs = stats.binom.pmf(outcomes, n_next_patients, p_disease)
most_likely = outcomes[np.argmax(probs)]

print(f"P(disease) per patient = {p_disease:.3f}")
print(f"Most likely count among the next {n_next_patients} patients screened: {most_likely}")
print(f"P(at least 15 of the next 20 have disease) = {stats.binom.sf(14, n_next_patients, p_disease):.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(outcomes, probs, color="darkorange")
ax.set_xlabel(f"patients with disease, out of next {n_next_patients}")
ax.set_title(f"Binomial(n={n_next_patients}, p={p_disease:.2f})")
plt.show()

### Poisson: staffing for a rare-event count

If the clinic wants to plan follow-up-cardiology-referral capacity, and referrals
happen at a roughly steady average rate per week, that's a Poisson question —
useful for the clinic's *operational* planning, distinct from the per-patient risk
score itself.

In [ ]:
referrals_per_disease_patient_per_year = 0.3  # illustrative operational rate
n_disease_patients = df["target"].sum()
avg_referrals_per_year = referrals_per_disease_patient_per_year * n_disease_patients

k = np.arange(0, int(avg_referrals_per_year) + 15)
pmf = stats.poisson.pmf(k, avg_referrals_per_year)
capacity_95pct = stats.poisson.ppf(0.95, avg_referrals_per_year)

print(f"Registry has {n_disease_patients} disease-positive patients")
print(f"Expected referrals/year at this rate: {avg_referrals_per_year:.1f}")
print(f"To handle referral volume in 95% of years, plan capacity for "
      f"{int(capacity_95pct)}, not just the average of {avg_referrals_per_year:.1f}.")

## Systems view — what this stage hands to the next one

Stage 1 (Notebooks 1-3) is complete: the system's output format (probability), its
inputs' central tendencies (Notebook 2), and now their distributional shapes are
all documented. Stage 2 shifts from *understanding data already collected* to
*designing how more of it should be collected* — Notebook 4 asks how the clinic
should sample new patients into this registry without quietly biasing every number
computed here.

## Try it yourself

1. Check `trestbps` (resting blood pressure) against a Normal distribution the same
   way `thalach` was checked — does it need a transform, or does it fit reasonably
   well as-is?
2. Using the Binomial model, find how many of the next 20 patients you'd need to
   see with disease before it would be surprising (say, below 5% probability) under
   the registry's current prevalence rate.
3. Increase `referrals_per_disease_patient_per_year` to 0.6 and recompute the 95th
   percentile capacity figure — how much does doubling the referral rate change the
   *staffing* recommendation, versus how much it changes the *average*?